In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('darkgrid')

os.makedirs('plots', exist_ok=True)
print("All libraries imported successfully!")

All libraries imported successfully!


In [41]:
df = pd.read_csv('../data/dataCleaned.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nPrice stats:")
print(df['price'].describe())
df.head()

Shape: (96125, 9)

Columns: ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'mpg', 'engineSize', 'brand']

Price stats:
count    96125.000000
mean     16567.395984
std       9472.798476
min       1000.000000
25%       9998.000000
50%      14200.000000
75%      20490.000000
max      99950.000000
Name: price, dtype: float64


,model,year,price,transmission,mileage,fuelType,mpg,engineSize,brand
0,A1,2017,12500,Manual,15735,Petrol,55.4,1.4,Audi
1,A6,2016,16500,Automatic,36203,Diesel,64.2,2.0,Audi
2,A1,2016,11000,Manual,29946,Petrol,55.4,1.4,Audi
3,A4,2017,16800,Automatic,25952,Diesel,67.3,2.0,Audi
4,A3,2019,17300,Manual,1998,Petrol,49.6,1.0,Audi


In [ ]:
bins   = [0, 10000, 20000, 100000]
labels = [0, 1, 2]
df['price_category'] = pd.cut(df['price'], bins=bins, labels=labels).astype(int)

label_names = {0: 'Budget', 1: 'Mid-Range', 2: 'Premium'}

print("Category distribution:")
counts = df['price_category'].value_counts().sort_index()
for cat, count in counts.items():
    print(f"  {label_names[cat]} ({cat}): {count:,} cars ({count/len(df)*100:.1f}%)")

colors = ['#4a9eff', '#7b61ff', '#34d399']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar([label_names[i] for i in sorted(counts.index)],
            [counts[i] for i in sorted(counts.index)],
            color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Price Category Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Cars')

axes[1].pie([counts[i] for i in sorted(counts.index)],
            labels=[label_names[i] for i in sorted(counts.index)],
            colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Price Category Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Target variable created!")

Category distribution:
  Budget (0): 24,981 cars (26.0%)
  Mid-Range (1): 46,472 cars (48.3%)
  Premium (2): 24,672 cars (25.7%)


In [ ]:
df_ml = df.copy()

# Car age
df_ml['car_age'] = 2020 - df_ml['year']
df_ml = df_ml.drop(columns=['year'])

# Log transform mileage
df_ml['mileage'] = np.log1p(df_ml['mileage'])

# One-hot encode
df_ml = pd.get_dummies(df_ml, columns=['brand', 'model', 'fuelType', 'transmission'], drop_first=False)

print("Shape after encoding:", df_ml.shape)

X = df_ml.drop(columns=['price', 'price_category'])
y = df_ml['price_category']

print("Features:", X.shape[1])
print("Target classes:", sorted(y.unique()))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ['mileage', 'engineSize', 'mpg', 'car_age']
scaler = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f"Training set:  {X_train.shape[0]:,} rows")
print(f"Test set:      {X_test.shape[0]:,} rows")
print(f"Features:      {X_train.shape[1]}")

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Select top 50 features
selector_50 = SelectKBest(f_classif, k=50)
X_train_50 = selector_50.fit_transform(X_train, y_train)
X_test_50  = selector_50.transform(X_test)

# Select top 20 features
selector_20 = SelectKBest(f_classif, k=20)
X_train_20 = selector_20.fit_transform(X_train, y_train)
X_test_20  = selector_20.transform(X_test)

# Show top 20 most important features
feature_scores = pd.Series(selector_50.scores_, index=X_train.columns)
top_features = feature_scores.nlargest(20)

plt.figure(figsize=(12, 6))
top_features.sort_values().plot(kind='barh', color='#4a9eff', edgecolor='white')
plt.title('Top 20 Most Important Features (ANOVA F-score)', fontweight='bold')
plt.xlabel('F-Score')
plt.tight_layout()
plt.savefig('plots/02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 features:")
for feat, score in feature_scores.nlargest(10).items():
    print(f"  {feat}: {score:.1f}")

configs = ['All Features', 'Top 50', 'Top 20']
print(f"\nFeature counts: All={X_train.shape[1]}, Top50=50, Top20=20")

In [ ]:
# Baseline Logistic Regression
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train, y_train)
lr_base_pred = lr_base.predict(X_test)

print("Baseline Logistic Regression Results:")
print(f"  Accuracy:  {accuracy_score(y_test, lr_base_pred):.4f}")
print(f"  Precision: {precision_score(y_test, lr_base_pred, average='weighted'):.4f}")
print(f"  Recall:    {recall_score(y_test, lr_base_pred, average='weighted'):.4f}")
print(f"  F1:        {f1_score(y_test, lr_base_pred, average='weighted'):.4f}")

In [ ]:
param_grid_lr = {
    'C':        [0.01, 0.1, 1, 10, 100],
    'solver':   ['lbfgs', 'saga'],
    'penalty':  ['l2'],
    'max_iter': [1000]
}

print("Running GridSearchCV for Logistic Regression...")
grid_lr = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid_lr,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
grid_lr.fit(X_train, y_train)

print(f"\nBest parameters: {grid_lr.best_params_}")
print(f"Best CV F1 score: {grid_lr.best_score_:.4f}")